# Bitcoin Event-Driven Price Impact Analysis with Hugging Face NLP

This notebook analyzes how real-world events affect Bitcoin price by:
- Extracting entities from news articles using Named Entity Recognition (NER)
- Classifying event types using zero-shot classification (e.g., Regulatory, Adoption, Market Manipulation)
- Aligning articles with Bitcoin price data from CoinGecko
- Measuring next-day price impact
- Visualizing event types and their price effect using Plotly

The goal is to understand how specific categories of news impact Bitcoin price trends.

In [2]:
!pip install lxml_html_clean


In [3]:
!pip install pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 57.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 37.8 MB/s eta 0:00:00 0:00:01


In [4]:
!pip install torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 MB 61.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 36.3 MB/s eta 0:00:00


In [5]:
!pip install torch --quiet

In [6]:
!pip install torch transformers newspaper3k pandas


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 50.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 783.6/783.6 kB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 74.1 MB/s eta 0:00:00
  Created wheel for tinysegmenter: filename=tinysegmenter-0.3-py3-none-any.whl size=13538 sha256=68f3e921f1978da4ce40ed500ff9e8a0d27af437c0a8833adfbfa08a3a13d8bb
  Stored in directory: /root/.cache/pip/wheels/99/74/83/8fac1c8d9c648cfabebbbffe97a889f6624817f3aa0bbe6c09
  Created wheel for feedfinder2: filename=feedfinder2-0.0.4-py3-none-any.whl size=3355 sha256=c17

  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6064 sha256=3a81db28cd76132d25efe9c6a3a2240858dfe257619e8f518eaee87e339477fd
  Stored in directory: /root/.cache/pip/wheels/83/63/2f/117884c3b19d46b64d3d61690333aa80c88dc14050e269c546
Successfully built tinysegmenter feedfinder2 jieba3k sgmllib3k


In [7]:
!pip install newspaper3k transformers

from newspaper import Article
from transformers import pipeline
import pandas as pd
import datetime


/usr/local/lib/python3.8/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Sample article data (manually inserted)
df_articles = pd.DataFrame([
    {
        "url": "https://example.com/news/sec-delays-etf",
        "title": "SEC Delays Decision on Bitcoin ETF Again",
        "text": "The U.S. SEC has once again delayed its decision on multiple Bitcoin ETF applications...",
        "date": datetime.date(2025, 4, 27)
    },
    {
        "url": "https://example.com/news/microstrategy-buys-more",
        "title": "MicroStrategy Acquires Additional 1,200 BTC",
        "text": "In its latest crypto move, MicroStrategy has added another 1,200 Bitcoin to its reserves...",
        "date": datetime.date(2025, 4, 28)
    },
    {
        "url": "https://example.com/article1",
        "text": "Bitcoin price jumped above $70,000 following the approval of a Bitcoin ETF by the SEC...",
        "date": datetime.date(2025, 4, 25)
    },
    {
        "url": "https://example.com/article2",
        "text": "BlackRock has officially added Bitcoin to its institutional portfolio...",
        "date": datetime.date(2025, 4, 20)
    },
    {
        "url": "https://example.com/article3",
        "text": "A major security breach on the Binance exchange has resulted in unauthorized withdrawals...",
        "date": datetime.date(2025, 4, 22)
    }
])


df_articles.head()


,url,title,text,date
0,https://example.com/news/sec-delays-etf,SEC Delays Decision on Bitcoin ETF Again,The U.S. SEC has once again delayed its decisi...,2025-04-27
1,https://example.com/news/microstrategy-buys-more,"MicroStrategy Acquires Additional 1,200 BTC","In its latest crypto move, MicroStrategy has a...",2025-04-28
2,https://example.com/article1,NaN,"Bitcoin price jumped above $70,000 following t...",2025-04-25
3,https://example.com/article2,NaN,BlackRock has officially added Bitcoin to its ...,2025-04-20
4,https://example.com/article3,NaN,A major security breach on the Binance exchang...,2025-04-22


In [9]:
!pip install transformers


In [10]:
import sys
!{sys.executable} -m pip install torch


In [11]:
import sys
!{sys.executable} -m pip install transformers

In [12]:
import sys
!{sys.executable} -m pip install torch

In [13]:
# NER
from transformers import pipeline

ner = pipeline("ner", model="dslim/bert-base-NER", grouped_entities=True)
df_articles["entities"] = df_articles["text"].apply(lambda x: ner(x[:512]))
df_articles[["title", "entities"]].head()



Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
/usr/local/lib/python3.8/dist-packages/transformers/pipelines/token_classification.py:170: UserWarning: `grouped_entities` is deprecated and will be removed in version v5.0.0, defaulted to `aggregation_strategy="simple"` instead.
  warnings.warn(


,title,entities
0,SEC Delays Decision on Bitcoin ETF Again,"[{'entity_group': 'ORG', 'score': 0.78774965, ..."
1,"MicroStrategy Acquires Additional 1,200 BTC","[{'entity_group': 'ORG', 'score': 0.9982812, '..."
2,NaN,"[{'entity_group': 'MISC', 'score': 0.47908634,..."
3,NaN,"[{'entity_group': 'ORG', 'score': 0.99895114, ..."
4,NaN,"[{'entity_group': 'ORG', 'score': 0.8766595, '..."


In [14]:
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
labels = ["Regulatory", "Technological", "Market Manipulation", "Adoption News"]

df_articles["event_type"] = df_articles["text"].apply(lambda x: classifier(x[:512], labels)["labels"][0])
df_articles[["title", "event_type"]].head()


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


,title,event_type
0,SEC Delays Decision on Bitcoin ETF Again,Regulatory
1,"MicroStrategy Acquires Additional 1,200 BTC",Adoption News
2,NaN,Adoption News
3,NaN,Adoption News
4,NaN,Technological


In [15]:
!pip install pycoingecko


In [16]:
from pycoingecko import CoinGeckoAPI
import pandas as pd

cg = CoinGeckoAPI()

# Get Bitcoin prices for the past 90 days
btc_data = cg.get_coin_market_chart_by_id(id='bitcoin', vs_currency='usd', days=90)

# Convert to DataFrame
df_prices = pd.DataFrame(btc_data['prices'], columns=['timestamp', 'price'])
df_prices['date'] = pd.to_datetime(df_prices['timestamp'], unit='ms').dt.date

# Clean output
df_prices = df_prices.drop('timestamp', axis=1)
df_prices = df_prices.groupby('date').mean().reset_index()
df_prices.head()


,date,price
0,2025-01-31,103858.818470
1,2025-02-01,101974.954741
2,2025-02-02,99107.359334
3,2025-02-03,96866.647778
4,2025-02-04,99368.634684


In [17]:
# Merge article events with Bitcoin daily price
df_merged = pd.merge(df_articles, df_prices, on='date', how='left')

# Show merged result
df_merged[["date", "title", "event_type", "price"]].head()


,date,title,event_type,price
0,2025-04-27,SEC Delays Decision on Bitcoin ETF Again,Regulatory,94170.938121
1,2025-04-28,"MicroStrategy Acquires Additional 1,200 BTC",Adoption News,94328.545246
2,2025-04-25,NaN,Adoption News,94246.061185
3,2025-04-20,NaN,Adoption News,84760.039451
4,2025-04-22,NaN,Technological,89583.332842


In [18]:
# Create a shifted price column for "next day"
df_prices_shifted = df_prices.rename(columns={"price": "price_next_day", "date": "date_prev"})
df_shifted = pd.merge(df_merged, df_prices_shifted, left_on="date", right_on="date_prev", how="left")

# Calculate delta (price change from day of event to next day)
df_shifted["price_change"] = df_shifted["price_next_day"] - df_shifted["price"]

# Show result
df_shifted[["date", "title", "event_type", "price", "price_next_day", "price_change"]].head()


,date,title,event_type,price,price_next_day,price_change
0,2025-04-27,SEC Delays Decision on Bitcoin ETF Again,Regulatory,94170.938121,94170.938121,0.0
1,2025-04-28,"MicroStrategy Acquires Additional 1,200 BTC",Adoption News,94328.545246,94328.545246,0.0
2,2025-04-25,NaN,Adoption News,94246.061185,94246.061185,0.0
3,2025-04-20,NaN,Adoption News,84760.039451,84760.039451,0.0
4,2025-04-22,NaN,Technological,89583.332842,89583.332842,0.0


In [19]:
!pip install plotly
!pip install ipython nbformat --upgrade


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 50.5 MB/s eta 0:00:00a 0:00:01


In [20]:
!pip install ipython nbformat --upgrade


In [21]:
import plotly.io as pio
print(pio.renderers)

Renderers configuration
-----------------------
    Default renderer: 'plotly_mimetype+notebook'
    Available renderers:
        ['plotly_mimetype', 'jupyterlab', 'nteract', 'vscode',
         'notebook', 'notebook_connected', 'kaggle', 'azure', 'colab',
         'cocalc', 'databricks', 'json', 'png', 'jpeg', 'jpg', 'svg',
         'pdf', 'browser', 'firefox', 'chrome', 'chromium', 'iframe',
         'iframe_connected', 'sphinx_gallery', 'sphinx_gallery_png']



In [23]:
import plotly.express as px

fig = px.scatter(
    df_shifted.dropna(subset=["price_change"]),
    x="date",
    y="price_change",
    color="event_type",
    hover_data=["title"],
    title="Bitcoin Price Change After Event (Next-Day Delta)"
)
import plotly.io as pio
pio.renderers.default = "iframe_connected"
fig.show()


## Summary: Event-Driven Bitcoin Price Impact Analysis (Checkpoint 1)

This notebook performs an event-driven analysis of Bitcoin price changes using the following pipeline:

- **NER and Event Classification:** Extracted named entities and classified event types using Hugging Face's `dslim/bert-base-NER` and `facebook/bart-large-mnli`
-  **Price Alignment:** Pulled historical Bitcoin prices via CoinGecko API and aligned them with article dates
-  **Impact Measurement:** Estimated short-term price impact (next-day delta) per event
-  **Visualization:** Generated an interactive Plotly scatter plot of events vs. price changes


